# Fine-Tuning vs. RAG vs. Prompting

## Why this matters

Every one of your real projects made this decision implicitly, without necessarily naming it as a decision. The X++ review agent uses pure prompting (a detailed system prompt per agent, no retrieval, no fine-tuning). The `local-rag-llm` PDF chatbot uses RAG (retrieval over your documents, base model unchanged). Neither project fine-tunes anything. This notebook makes explicit *why* those were the right calls, and what would have to be true for fine-tuning to actually be the right lever instead — because "fine-tune it" is the answer interviewers expect people to reach for too early, and knowing why you didn't is a stronger signal than knowing how to do it.

## Level 1: What it is

Three different levers for getting an LLM to do what you want, solving three different problems:

1. **Prompting** — instructions in the context window, every call. Changes *behavior/format/task-framing*. Nothing is persisted between calls; every request re-teaches the model what to do.
2. **RAG (Retrieval-Augmented Generation)** — inject *relevant external knowledge* into the context window at call time, retrieved from a store (vector DB, search index) based on the query. Changes *what facts the model has access to*, not how it behaves.
3. **Fine-tuning** — additional training on your own examples, producing a modified model (or a small adapter layer, in the common LoRA/PEFT approach) that's used instead of / alongside the base model. Changes *the model's default behavior itself*, persisted into the weights, no longer needing to be re-specified in every prompt.

## Level 2: How it actually works internally

### Prompting

The system prompt and user message are just tokens in the context window, exactly like any other input. The model's weights never change; what changes is the specific conditional distribution it's sampling from, because you've changed what it's conditioning on. This is why the X++ agents' system prompts are as detailed as they are ("1. SELECT INSIDE LOOP (Critical) -- Any select statement inside a while, for, or do-while loop...") -- every instruction has to be re-supplied every single call, because nothing persists.

**Cost model:** input tokens (the prompt) are charged every call. A long, detailed system prompt is a recurring cost, not a one-time cost.

### RAG

Covered in depth in the `rag-and-retrieval` track once we get there, but the internal mechanism in one sentence: a retriever (FAISS, BM25, or hybrid, as in `local-rag-llm`) finds the most relevant chunks of your document store for the current query, and those chunks get concatenated into the prompt alongside the user's question, before the model ever sees it. The model itself is completely unchanged -- it's just being given different *input* each time, sourced dynamically rather than hand-written.

**Cost model:** retrieval step (cheap, no LLM call) + input tokens for however many chunks got retrieved and stuffed into context (this scales with chunk count and size, and is a recurring per-call cost like prompting).

### Fine-tuning

You supply a training set of (input, desired output) pairs. Full fine-tuning updates all the model's weights via gradient descent on your examples -- extremely expensive and rarely done to production LLMs by anyone outside the model provider itself. The dominant practical approach is **LoRA (Low-Rank Adaptation)**: instead of updating the full weight matrices, you freeze the original weights and train a small pair of low-rank matrices that get added on top at inference time. This is dramatically cheaper (a LoRA adapter might be megabytes, versus the base model's tens of gigabytes) and lets you swap adapters in and out without needing a separate copy of the full model per fine-tune.

**Cost model:** large one-time training cost (compute + your labeled data), then *near-zero marginal cost per call* -- you're no longer paying for a long instructional prompt every time, because the behavior is baked into the weights.

## Level 3: Where it breaks, and the actual decision framework

The honest decision tree, in the order you should actually check things:

1. **Is the problem "the model doesn't know something"?** -> RAG, not fine-tuning. This is the single most common mistake: people fine-tune to "teach" a model facts, when fine-tuning is bad at reliably encoding specific facts (the model can still hallucinate details even after fine-tuning on them) and RAG is *architecturally* the right tool for grounding output in specific, retrievable, updatable facts. Your `local-rag-llm` project is exactly this case -- the documents change, so the knowledge needs to stay retrievable and swappable, not baked into weights that would need retraining every time a PDF is added.

2. **Is the problem "the model doesn't behave/format the way I need"?** -> Prompting first, always. It's free to iterate (change a system prompt, re-run, see the difference in seconds) versus fine-tuning (collect a dataset, train, evaluate, and you can't easily "undo" a bad fine-tune the way you can revert a prompt). The X++ project's detailed per-agent system prompts are this case: precise behavioral instructions, no training needed, and you were still iterating on wording days into the project (trimming the LLM prompts after moving checks to deterministic code) -- that kind of rapid iteration is only possible because it's just text, not a training run.

3. **Only fine-tune when prompting has genuinely failed *and* the task is stable.** Real trigger conditions: (a) the behavior needs to be extremely consistent across huge call volume, and the recurring token cost of a long system prompt is actually a meaningful expense at that scale; (b) you need the model to reliably do something a prompt can't teach at all -- like adopting a very specific, unusual output style that's hard to fully specify in words but easy to demonstrate with hundreds of examples; (c) latency matters enough that removing a long system prompt's input tokens meaningfully speeds up response time.

4. **Fine-tuning doesn't fix hallucination, and this is the most-tested interview question in this space.** A fine-tuned model can still confidently state something false -- fine-tuning shapes *style and behavior*, not *factual grounding*. If the real problem is "the model makes things up," the fix is RAG (give it real facts to cite) or better prompting (explicit grounding instructions, refuse-if-unsure framing), not fine-tuning. Conflating "model doesn't know X" with "model needs fine-tuning" is the single most common wrong answer to this topic.

5. **RAG and fine-tuning are not mutually exclusive, and "combine them" is a real answer, not a hedge.** A fine-tuned model can still use RAG for facts while the fine-tuning handles domain-specific *reasoning style* (e.g., a legal-document model fine-tuned to reason the way a lawyer does, while still retrieving the actual case law via RAG rather than relying on memorized training data). Don't present these as strictly either/or in an interview -- the strongest answer names the actual production combination when it applies.

6. **The X++ project specifically: why not fine-tune the Security/Performance agents to reduce judgment variance?** Worth having a sharp answer ready, since it's the natural next question given everything discussed in that project. Fine-tuning could plausibly make the model's judgment calls *more consistent* with a labeled training set of "here's what a senior reviewer would flag." But it wouldn't fix the actual root problem discussed at length in that project: there's no ground-truth dataset to fine-tune against yet (no compiled/verified vulnerability labels), and building one *is* the actual bottleneck -- fine-tuning without labeled ground truth just bakes in whatever bias exists in whatever informal examples you'd use, with none of the transparency a documented prompt has. This is a case where the real blocker (compiler/environment access, discussed in the X++ sessions) has to be solved first, and fine-tuning wouldn't bypass it.

In [ ]:
# Illustrating the cost-model difference discussed above -- not a real
# fine-tuning call (that's a separate training pipeline, not a
# messages.create() call), just making the recurring-cost point concrete.

# PROMPTING: this ~600-token system prompt is billed as input tokens
# on EVERY single call, forever.
SECURITY_SYSTEM_PROMPT_EXCERPT = """
You are a senior Microsoft Dynamics 365 FnO / AX security architect...
1. DIRECT SQL INJECTION ...
2. MISSING SysEntryPoint ...
3. MISSING ACCESS CHECKS ...
"""  # (truncated -- the real one is much longer)

approx_tokens_per_call = len(SECURITY_SYSTEM_PROMPT_EXCERPT.split()) * 1.3  # rough estimate
calls_per_day = 200
print(f"Recurring prompt overhead: ~{approx_tokens_per_call:.0f} tokens x {calls_per_day} calls/day")
print(f"= ~{approx_tokens_per_call * calls_per_day:.0f} tokens/day JUST for re-stating instructions")

# FINE-TUNING: after training, that same behavioral instruction is baked
# into the weights. The equivalent call needs little to no system prompt
# re-statement -- the marginal per-call token cost drops, but only after
# paying the (much larger, one-time) training cost up front. Whether that
# trade pays off depends entirely on call volume -- at 200 calls/day for
# a personal project, it almost never does.

## Interview-ready summary

**Q: "When would you fine-tune a model instead of using RAG or better prompting?"**

Weak answer: "If prompting isn't accurate enough, I'd fine-tune it on more examples."

Strong answer: "Almost never as a first move. If the problem is the model lacking specific, updatable facts, that's RAG's job, not fine-tuning's -- fine-tuning is unreliable for encoding precise facts and doesn't solve hallucination, it shapes behavior and style, not factual grounding. If the problem is behavior or output format, I'd iterate on prompting first, since it's free to change and instantly testable, versus fine-tuning which requires a labeled dataset and a training run before you even know if it worked. I'd only reach for fine-tuning once prompting has genuinely plateaued *and* there's a real, stable, high-volume use case where the recurring token cost of a long system prompt or the need for a very specific style that's hard to fully specify in words justifies the upfront training cost. And they're not mutually exclusive -- a fine-tuned model can still use RAG for the facts it cites."